In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score
)

Vamos baixar as bases de treino e teste para trabalharmos

In [ ]:
pasta_saida = Path("dados/processed/splits")

x_treino_smote = pd.read_csv(pasta_saida / "x_treino_smote.csv")
x_treino_scaled_smote = pd.read_csv(pasta_saida / "x_treino_scaled_smote.csv")

x_teste = pd.read_csv(pasta_saida / "x_teste.csv")
x_teste_scaled = pd.read_csv(pasta_saida / "x_teste_scaled.csv")

y_treino_smote = pd.read_csv(pasta_saida / "y_treino_smote.csv")["internacao_prolongada"]
y_treino_scaled_smote = pd.read_csv(pasta_saida / "y_treino_scaled_smote.csv")["internacao_prolongada"]

y_teste = pd.read_csv(pasta_saida / "y_teste.csv")["internacao_prolongada"]


Criamos uma função para salvar os resultados dessa fase

In [ ]:
resultados_ajustado = []

def salvar_resultados_ajustado(nome_modelo, y_real, y_pred): 
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred).ravel()
    
    resultados_ajustado.append({
        'Modelo': nome_modelo,
        'Accuracy': accuracy_score(y_real, y_pred),
        'Recall': recall_score(y_real, y_pred),
        'Precision': precision_score(y_real, y_pred),
        'F1-score': f1_score(y_real, y_pred),
        'Verdadeiros Negativos': tn,
        'Falsos Positivos': fp,
        'Falsos Negativos': fn,
        'Verdadeiros Positivos': tp
    })

Nessa fase usaremos os 3 modelos com melhor desempenho na fase anterior, que são:

1. Regressão Logística com SMOTE: teve o melhor f1-score da tabela e o segundo melhor recall sendo o modelo com melhor equilibrio geral entre encontrar casos positivos e manter uma precisão aceitavel

2. Random Forest com SMOTE: Segundo melhor f1-score e melhor precisão entre os modelos com SMOTE tendo o desempenho mais equilibrado que o XGBoost SMOTE

3. XGBoost com SMOTE: Melhor recall da tabela, indentificando mais casos positivos mesmo com a precisão menor, valendo a pena avançar pois reduz bastante os falsos negativos

eu irei fazer um ajuste de hiperparametros que eu considero o basico para cada modelo, e apos observar seus respectivos desempenhos, leveramos o modelo para a ultima fase onde realizaremos um ajuste mais pesado de hiperparametros visando ter um melhor desempenho

# Regressão Logística com SMOTE Ajustado

In [3]:
modelo_reglog_ajustado = LogisticRegression(
    C=0.5,
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

modelo_reglog_ajustado.fit(x_treino_scaled_smote, y_treino_scaled_smote)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.5
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

Fazendo previsões

In [4]:
y_pred_reglog_smote_ajustado = modelo_reglog_ajustado.predict(x_teste_scaled)

Avaliamos o modelo

In [5]:
print('Matriz de confusão:')
print(confusion_matrix(y_teste, y_pred_reglog_smote_ajustado))

print('\nClassification Report:')
print(classification_report(y_teste, y_pred_reglog_smote_ajustado))

print('Accuracy:', accuracy_score(y_teste, y_pred_reglog_smote_ajustado))
print('Recall:', recall_score(y_teste, y_pred_reglog_smote_ajustado))
print('Precision:', precision_score(y_teste, y_pred_reglog_smote_ajustado))
print('F1-score:', f1_score(y_teste, y_pred_reglog_smote_ajustado))

Matriz de confusão:
[[160249  43442]
 [ 14530  40157]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.79      0.85    203691
           1       0.48      0.73      0.58     54687

    accuracy                           0.78    258378
   macro avg       0.70      0.76      0.71    258378
weighted avg       0.82      0.78      0.79    258378

Accuracy: 0.7756310521793651
Recall: 0.7343061422275861
Precision: 0.48035263579707893
F1-score: 0.5807818578887234


O modelo apresentou recall d aproximadamente 73,34% para a classe 1, identificando corretamente 40107 das 54687 internações prolongadas.

comparada com a versão anterior sem ajuste, houve uma pequena melhora no recall, indicando que o modelo ficou ligeiramente mais sensivel a detectar internações prolongadas apesar dessa melhora vir acompanhada de uma leve redução na precision e no f1-score

a precision para a classe 1 foi de aproximadamente 48,08%, indicando que entre os casos classificados como internação prolongada, cerca de 48% realmente pertenciam a classe 1. já o f1-score foi de aproximadamente 58,08%, valor muito proximo ao modelo com configuração padrão

de forma geral os ajustes não trouxeram ganho expressivo ao modelo, por isso essa versão pode ser considerada interessante caso o objetivo seja priorizar a detecção de internações prolongadas, mas não representa uma melhora clara no equilibrio geral do modelo

In [11]:
salvar_resultados_ajustado('Regressão Logística Ajustada', y_teste, y_pred_reglog_smote_ajustado)

In [12]:
resultados_modelos_ajustados = pd.DataFrame(resultados_ajustado)
resultados_modelos_ajustados

,Modelo,Accuracy,Recall,Precision,F1-score,Verdadeiros Negativos,Falsos Positivos,Falsos Negativos,Verdadeiros Positivos
0,Regressão Logística Ajustada,0.775631,0.734306,0.480353,0.580782,160249,43442,14530,40157


# Random Forest com SMOTE ajustado

In [6]:
modelo_random_forest_smote_ajustado = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

modelo_random_forest_smote_ajustado.fit(x_treino_smote, y_treino_smote)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",250
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",20
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

agora fazemos algumas previsões

In [7]:
y_pred_random_forest_smote_ajustado = modelo_random_forest_smote_ajustado.predict(x_teste)

Agora avaliamos o modelo

In [8]:
print('Matriz de confusão:')
print(confusion_matrix(y_teste, y_pred_random_forest_smote_ajustado))

print('\nClassification Report:')
print(classification_report(y_teste, y_pred_random_forest_smote_ajustado))

print('Accuracy:', accuracy_score(y_teste, y_pred_random_forest_smote_ajustado))
print('Recall:', recall_score(y_teste, y_pred_random_forest_smote_ajustado))
print('Precision:', precision_score(y_teste, y_pred_random_forest_smote_ajustado))
print('F1-score:', f1_score(y_teste, y_pred_random_forest_smote_ajustado))

Matriz de confusão:
[[133773  69918]
 [ 12986  41701]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.66      0.76    203691
           1       0.37      0.76      0.50     54687

    accuracy                           0.68    258378
   macro avg       0.64      0.71      0.63    258378
weighted avg       0.80      0.68      0.71    258378

Accuracy: 0.6791367686103306
Recall: 0.7625395432186809
Precision: 0.3736012686012238
F1-score: 0.5014972400274194


O modelo apresentou um aumento expressivo da classe 1, identificando 42324 das 54687 internações prolongadas presentes, tendo um recall de 77,39%, indicando que o modelo passou a detectar uma proporção relativamente maior dos casos de internações prolongadas

apesar disso, essa melhora veio seguida de uma queda na metrica precision da classe 1, causado pelo aumeto de falsos positivos obtendo agora 65708 casos, ou seja ele classificou muitos internações não prolongadas como prolongadas

o f1-score da classe 1 foi de aproximadamenre 52,02%, levemente inferior ao resultado obtido pelo modelo sem ajustes indicando um aumento de desequilibrio entre recall e precision

em suma, o modelo se tornou mais sensivel a classe de interesse, mas tambem aumentou excessivamente os falsos positivos, com isso não superando sua versão não ajustada

In [13]:
salvar_resultados_ajustado('Random Forest Ajustado', y_teste, y_pred_random_forest_smote_ajustado)

In [14]:
resultados_modelos_ajustados = pd.DataFrame(resultados_ajustado)
resultados_modelos_ajustados

,Modelo,Accuracy,Recall,Precision,F1-score,Verdadeiros Negativos,Falsos Positivos,Falsos Negativos,Verdadeiros Positivos
0,Regressão Logística Ajustada,0.775631,0.734306,0.480353,0.580782,160249,43442,14530,40157
1,Random Forest Ajustado,0.679137,0.762540,0.373601,0.501497,133773,69918,12986,41701


# XGBoost com SMOTE Ajustado

In [15]:
modelo_xgboost_smote_ajustado = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.07,
    min_child_weight=5,
    gamma=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2,
    reg_alpha=0,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

modelo_xgboost_smote_ajustado.fit(x_treino_smote, y_treino_smote)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


Agora fazemos algumas previsões

In [16]:
y_pred_xgboost_smote_ajustado = modelo_xgboost_smote_ajustado.predict(x_teste)

Agora avaliamos o modelo

In [17]:
print('Matriz de confusão:')
print(confusion_matrix(y_teste, y_pred_xgboost_smote_ajustado))

print('\nClassification Report:')
print(classification_report(y_teste, y_pred_xgboost_smote_ajustado))

print('Accuracy:', accuracy_score(y_teste, y_pred_xgboost_smote_ajustado))
print('Recall:', recall_score(y_teste, y_pred_xgboost_smote_ajustado))
print('Precision:', precision_score(y_teste, y_pred_xgboost_smote_ajustado))
print('F1-score:', f1_score(y_teste, y_pred_xgboost_smote_ajustado))

Matriz de confusão:
[[142807  60884]
 [ 12553  42134]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.70      0.80    203691
           1       0.41      0.77      0.53     54687

    accuracy                           0.72    258378
   macro avg       0.66      0.74      0.66    258378
weighted avg       0.81      0.72      0.74    258378

Accuracy: 0.7157768850289111
Recall: 0.7704573298955876
Precision: 0.4089964860509814
F1-score: 0.5343394312165118


O modelo identificou 42134 das 54687 internações prolongadas, resultando em um recall de aproximadamente 77,05% para a classe 1

apesar disso a versão ajustada do modelo apresentou queda em relação ao modelo original tendo as 3 metricas principais inferiores ao modelo anterior

a precision da classe 1 ficou em aproximadamenre 40,90%, indicando que ele gerou muitos falsos positivos mostrando que o modelo continuou sensivel a classe 1, mas com baixa capacidade de separar corretamente os casos prolongados dos não prolongados

o f1-score da classe 1 foi aproximadamente 53,43%, inferior ao resultado obtido pelo modelo não ajustado, ou seja apesar do recall elevado, o ajuste de hiperparametros não melhorou o equilibrio entre recall e precision

em suma, a versão ajustada do modelo é inferior ao modelo original

In [18]:
salvar_resultados_ajustado('XGBoost Ajustado', y_teste, y_pred_xgboost_smote_ajustado)

In [19]:
resultados_modelos_ajustados = pd.DataFrame(resultados_ajustado)
resultados_modelos_ajustados

,Modelo,Accuracy,Recall,Precision,F1-score,Verdadeiros Negativos,Falsos Positivos,Falsos Negativos,Verdadeiros Positivos
0,Regressão Logística Ajustada,0.775631,0.734306,0.480353,0.580782,160249,43442,14530,40157
1,Random Forest Ajustado,0.679137,0.762540,0.373601,0.501497,133773,69918,12986,41701
2,XGBoost Ajustado,0.715777,0.770457,0.408996,0.534339,142807,60884,12553,42134


Com isso podemos observar que

1. Regressão Logistica ajustada teve o melhor desempenho geral entre modelos avaliados. apesar de não ter obtido o maior recall, apresentou o mais f1-score e a melhor precision, mantendo ainda uma boa capacidade de identificar internações prolongadas nos indicando que o modelo conseguiu equilibrar melhor a detecção da classe de interesse com uma quantidade menor de falsos positivos.

2. Random Forest ajustado apresentou recall elevado, identificando uma quantidade relevante de internações prolongadas. porem, sua precision foi a mais baixa entre os 3 modelos, indicando que ele classificou muitos casos não prolongados como prolongados. como consequencia, seu f1-score foi inferior ao dos demais modelos, mostrando menor equilibrio entre recall e precision

3. XGBoost ajustado apresentou maior recall entre os modelos avaliados, sendo o que mais identificou corretamente internações prolongadas. porem, sua precision foi inferior à da Regressão Logistica ajustada, indicando maior numero de falsos positivos. ainda assin, seu desempenho foi superior ao Random Forest ajustado em f1-score, tornando uma opção interessante para ajustes posteriores

em suma, Regressão logistica ajustada foi o melhor modelo nesta etapa, levando em conta o equilibrio entre recall, precision e f1-score. O XGBoost ajustado teve alto recall e apresenta uma maior possibilidade de ajustes de hiperparametros, permanecendo como melhor candidato para a etapa de ajustes mais finos. já o Random Forest ajustado teve um desempenho pior em equilibrio geral, principalmente devido a baixa precision e ao elevado numero de falsos positivos

com isso em mente, embora a Regressão Logistica ajustada tenha um melhor desempenho geral nesta etapa, com maior f1-score, ou seja, equilibrio geral entre recall e precision, decidimos levar a frente o XGBoost ajustado para a proxima fase de ajuste de hiperparametros

essa escolha foi tomada pelo fato do XGBoost apresentar maior quantidade de hiperparametros capazes de controlar a complexidade do modelo, a profundidade das arvores, a criação de novas divisões e a regularização, apresentando maior potencial de melhoria por meio de ajuste fino.

Assim sendo, a proxima etapa terá como objetivo verificar se é possivel reduzir os falsos positivos e melhorar a precision e o f1-score do xgboost, sem perder excessivamente o recall da classe 1. a Regressão Logistica Ajustada será usada como modelo de referencia, por ter sido o melhor modelo até o momento em termos de equilibrio geral.

In [21]:
# vou salvar a tabela com os resultados tambem
resultados_modelos_ajustados.to_csv("dados/processed/resultados/resultados_modelos_ajustados.csv", index=False)